<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/03_stance_detection_NLI_classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [ ]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import matthews_corrcoef

In [ ]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

In [ ]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/gold_annotated.csv')

In [ ]:
adj_to_base = {
    1:'positiv',
    2:'negativ',
    3:'neutral',
}

# Apply mapping
subsample_gold['gold_standard'] = subsample_gold['gold_standard'].map(adj_to_base)

In [ ]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

In [ ]:
subsample_gold.head()

##**Define Metrics**

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

#**Evaluate NLI Classifiers**


#**MoritzLaurer/bge-m3-zeroshot-v2.0**

In [ ]:
classifier = pipeline("zero-shot-classification", model='MoritzLaurer/bge-m3-zeroshot-v2.0')



###**Annotate Title + Paragraph**

In [ ]:
	samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + "Paragraph:\n" + subsample_gold['paragraph'])
	template = "Der Text impliziert eine {} Meinung zu Migration in Bezug auf Fachkräftemangel."
	labels = ['positive', 'negative', 'keine']

In [ ]:
# classify the documents
res = classifier(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our data frame
subsample_gold['label_ml_p6'] = [label['labels'][0] for label in res]

subsample_gold['score_ml_p6'] = [r['scores'][0] for r in res]

In [ ]:
adj_to_base = {
    'positive': 'positiv',
    'negative': 'negativ',
    'keine': 'neutral',
}

# Apply mapping
subsample_gold['label_ml_p6'] = subsample_gold['label_ml_p6'].map(adj_to_base)

In [ ]:
metrics_ml = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_ml_p6'])

print("Evaluation Metrics: MoritzLaurer/bge-m3-zeroshot-v2.0")
for metric, value in metrics_ml.items():
    print(f"{metric}: {value:.3f}")

#**mlburnham/Political_DEBATE_DeBERTa_large_v1.1**

In [ ]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1')

###**Annotate Title + Paragraph**

In [ ]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + subsample_gold['paragraph'])
template = 'Der Text positioniert sich {} zu Migration.'
labels = ['positiv', 'negativ', 'neutral']

In [ ]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our data frame
subsample_gold['label_mlb_p8'] = [label['labels'][0] for label in res]

subsample_gold['score_mlb_p8'] = [r['scores'][0] for r in res]

In [ ]:
metrics_mlb = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_mlb_p8'])

print("Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1")
for metric, value in metrics_mlb.items():
    print(f"{metric}: {value:.3f}")

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_mlb_p8']))

#**luissattelmayer/NLI-stance-immigration-burnham-finetuned-german**

In [ ]:
classifier_ls = pipeline ('zero-shot-classification', model='luissattelmayer/NLI-stance-immigration-burnham-finetuned-german')


###**Annotate Title + Paragraph**

In [ ]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + "Paragraph:\n" + subsample_gold['paragraph'])
template = "Der Text impliziert eine {} Meinung zu Migration in Bezug auf Fachkräftemangel."
labels = ['positive', 'negative', 'keine']

In [ ]:
# classify the documents
res = classifier_ls(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our data frame
subsample_gold['label_ls_p6'] = [label['labels'][0] for label in res]

subsample_gold['score_ls_p6'] = [r['scores'][0] for r in res]

In [ ]:
adj_to_base = {
    'positive': 'positiv',
    'negative': 'negativ',
    'keine': 'neutral',
}

# Apply mapping
subsample_gold['label_ls_p6'] = subsample_gold['label_ls_p6'].map(adj_to_base)

In [ ]:
metrics_ls = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_ls_p6'])

print("Evaluation Metrics: luissattelmayer/NLI-stance-immigration-burnham-finetuned-german")
for metric, value in metrics_ls.items():
    print(f"{metric}: {value:.3f}")

#**Evaluation**

In [ ]:
metrics_df = pd.DataFrame([metrics_ml, metrics_mlb, metrics_ls],
                          index=['MoritzLaurer', 'mlburnham', 'luissattelmayer'])

print(metrics_df.round(3))

###**Save Results**

In [ ]:
subsample_gold[['date', 'title', 'paragraph', 'newspaper', 'gold_standard', 'label_ml_p6', 'score_ml_p6', 'label_mlb_p8','score_mlb_p8', 'label_ls_p6', 'score_ls_p6' ]].to_csv('/content/drive/MyDrive/FKM/stance_detection/classification_results/results_NLI_classification.csv', index=False)